# Ablation V6 — Full PGA + Heatmap Prompt (Reference)
Cài đặt gốc (PSG+CAD+Gaussian heatmap) — kết quả phải ~Dice 0.86. Đây chính là mô hình "PGA-UNet (đề xuất)" dùng xuyên suốt khóa luận; setup/test lấy từ `pga-vs-unet2d-r512.ipynb`, phần hiển thị ảnh theo đúng style V1-V5 để dễ so sánh trực quan.

**Chạy tuần tự: Setup → Config → Test → Visualization**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (PGA-UNet, V6 — Full PGA + Heatmap, reference)
# ══════════════════════════════════════════════════════
import os, sys, gdown, torch

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = 'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'

if not os.path.exists(f'{BASE}/pga-repo'):
    os.system(f'git clone -q -b TN_B_ON {REPO} {BASE}/pga-repo')
print('  ✅ pga-repo')

DS_ZIP = f'{BASE}/dataset_BTXRD.zip'
if not os.path.exists(DS_ZIP):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   DS_ZIP, quiet=False)
if not os.path.exists(f'{BASE}/pga-repo/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_ZIP} -d {BASE}/pga-repo/')
print('  ✅ Dataset ready')

os.makedirs(f'{BASE}/pga-repo/checkpoints', exist_ok=True)
CKPT_PATH = f'{BASE}/pga-repo/checkpoints/pga_unet_expB_best.pth'
if not os.path.exists(CKPT_PATH):
    gdown.download('https://drive.google.com/uc?id=1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY',
                   CKPT_PATH, quiet=False)
print(f'  ✅ checkpoint  {os.path.getsize(CKPT_PATH)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Setup DONE  |  device={"cuda" if torch.cuda.is_available() else "cpu"}')


In [ ]:
USE_ENCODER_PROMPT = True   # PSG + CAD (mô hình gốc — không override class)
BINARY_PROMPT      = False  # plateau heatmap (default training setup)
VARIANT_NAME       = 'V6 — Full PGA + Heatmap (reference, should ~= Dice 0.86)'


In [ ]:
# ══════════════════════════════════════════════════════
# PHẦN 1 — PGA-UNet Test (IMG_SIZE=512, 3 prompt modes)
# ══════════════════════════════════════════════════════
import os, csv, sys
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/pga-repo'

# ── Clear module cache rồi load PGA modules ──────────────────────────
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del sys.modules[_k]
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
else: sys.path.remove(PGA_ROOT); sys.path.insert(0, PGA_ROOT)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR  = f'{PGA_ROOT}/dataset_BTXRD/test/images'
JSON_DIR = f'{PGA_ROOT}/dataset_BTXRD/test/annotations'
CKPT_PATH= f'{PGA_ROOT}/checkpoints/pga_unet_expB_best.pth'
os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ PGA-UNet loaded  [{DEVICE}]')

def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_metrics_img(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=calc_hd95(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.0
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']
all_image_records = {}
pga_csv_rows = []
pga_results  = {}   # dùng ở cell tổng hợp

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i, (img_t, mask_t, prompt_t) in enumerate(tqdm(loader, desc=f'[{mode}]')):
        img_name, _ = ds.all_samples[i]
        gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                  prob_max=prob.copy(),prompts=[prompt_np])
        else:
            g=groups[img_name]
            np.maximum(g['gt_union'],gt_np,out=g['gt_union'])
            np.maximum(g['prob_max'],prob, out=g['prob_max'])
            g['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=calc_metrics_img(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],
                             n_samples=len(g['prompts']),**m))
    all_image_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    pga_results[mode]=m_avg
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    pga_csv_rows.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

bar='='*82
print(f'\n{bar}\n  PGA-UNet — Image-level metrics (GT union+max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in pga_csv_rows:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{PGA_ROOT}/results/pga_unet2d_test_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(pga_csv_rows)
print(f'\n✅ CSV: {PGA_ROOT}/results/pga_unet2d_test_results.csv')

# ── Per-image CSV (phục vụ kiểm định thống kê Wilcoxon nếu cần, C.1) ─────────
with open(f'{PGA_ROOT}/results/pga_unet2d_per_image.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['mode', 'img_name'] + KEYS)
    for mode in MODES:
        for r in all_image_records[mode]:
            w.writerow([mode, r['img_name']] + [f"{r[k]:.6f}" for k in KEYS])
print(f'✅ CSV per-image saved: {PGA_ROOT}/results/pga_unet2d_per_image.csv')


In [ ]:
# ── Visualization: ≥10 ảnh có từ 2 GT polygon trở lên ───────────────────────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Chạy cell Test trước'

N_MIN  = 10
MIN_GT = 2

recs   = [r for r in all_image_records['zoom_out'] if r['n_samples'] >= MIN_GT]
if len(recs) < N_MIN:
    recs = all_image_records['zoom_out'][:N_MIN]
N_SHOW = len(recs)

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'{VARIANT_NAME} — {N_SHOW} ảnh (ưu tiên ≥{MIN_GT} polygon, zoom_out, image-level)',
             fontsize=12, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np  = rec['img'] * 0.5 + 0.5
    gt_np   = (rec['gt']   > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp=(pred_np*gt_np).sum(); fp=(pred_np*(1-gt_np)).sum(); fn=((1-pred_np)*gt_np).sum(); e=1e-6
    dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp+e)/(tp+fp+fn+e))
    pre=float((tp+e)/(tp+fp+e));         rec_=float((tp+e)/(tp+fn+e))
    n = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    pr_ov = bg.copy()
    pr_ov[...,0] = np.clip(pr_ov[...,0] + pred_np*0.55, 0, 1)
    pr_ov[...,1] = np.clip(pr_ov[...,1] - pred_np*0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    gt_ov = bg.copy()
    gt_ov[...,1] = np.clip(gt_ov[...,1] + gt_np*0.55, 0, 1)
    gt_ov[...,0] = np.clip(gt_ov[...,0] - gt_np*0.2,  0, 1)
    row[3].imshow(gt_ov)

    inter = bg.copy() * 0.25
    inter[...,1] = np.clip(inter[...,1] + pred_np*gt_np*0.9,      0, 1)
    inter[...,0] = np.clip(inter[...,0] + pred_np*(1-gt_np)*1.0,  0, 1)
    inter[...,2] = np.clip(inter[...,2] + (1-pred_np)*gt_np*1.0,  0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row: ax.axis('off')

fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),
                    Patch(facecolor='blue',label='FN')],
           loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5,-0.005))
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)
